**Author**: Adrian Vanyi

# Solver for the constrained optimization problem

This notebooks runs the solver to estimate the optimzal portfolio weights to optimization problem described in section §9 of the documentation (for either one of the variants `max_return` or `min_variance`)

**Prerequisite:** the data files (S&P 500 historical members, prices, dividends, SPY, EFFR) must already be in `../data/`. If they are not, run `python download_data.py` from the project root first.

### Notebook structure:

0. Setup
1. Load pre-downloaded data (and filter out bad tickers)
2. Compute risk-free returns
3. Configure the backtest calendar
4. Build per-rebalance-date universes
5. Configure the optimization problem
6. Solve the optimzation problem at the selected backtest rebalance date
7. Results

## 0. Setup

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
OUTPUTS_DIR.mkdir(exist_ok=True)

from modules import logging_setup
from modules import backtest_calendar as bcal
from modules import beta_computation as bc
from modules import covariance_matrix as cm
from modules import factors_engineering as fe
from modules import rates as r
from modules import returns_prediction_model as rpm
from modules import universe_construction as uc
from modules import market_data as md
from modules import strategies as strat
from modules import weights_optimizer as wo


logging_setup.configure(level="INFO", stream = sys.stdout)

## 1. Load pre-downloaded data (and filter out bad tickers)
*(The filtering out of bad tickers is explained in §6.5 and Appendix 1 of the documentation*.)


In [3]:
expected_files = [
    "sp500_members_at_start_of_months.parquet",
    "prices_sp500_members.parquet",
    "dividends_data.parquet",
    "daily_spy_returns.parquet",
    "EFFR.csv",
]
missing = [f for f in expected_files if not (DATA_DIR / f).exists()]
if missing:
    raise FileNotFoundError(
        f"data files missing in {DATA_DIR}: {missing}. "
        f"Run `python download_data.py` from the project root first."
    )

# S&P 500 monthly historical members (DataFrame: index=first trading day of
# month, columns are member tickers, eventually extended with NaN to form a uniform width).
sp500_members_at_first_trading_day_of_every_month = pd.read_parquet(
    DATA_DIR / "sp500_members_at_start_of_months.parquet"
)
first_trading_day_of_month_to_sp500_members_dict = {
    idx: row.dropna().tolist()
    for idx, row in sp500_members_at_first_trading_day_of_every_month.iterrows()
}

# Daily price panel for all S&P 500 members during the data window
price_data = pd.read_parquet(DATA_DIR / "prices_sp500_members.parquet")

# Dividend events (long Series indexed by (ticker, ex-dividend date) and values the declared dividend-per-share)
dividend_data = pd.read_parquet(DATA_DIR / "dividends_data.parquet").squeeze()

# SPY adjusted-close prices, used as the market proxy for beta estimation
daily_spy_returns = pd.read_parquet(DATA_DIR / "daily_spy_returns.parquet").squeeze()

# EFFR (Effective Fed Funds Rate), the daily risk-free rate
effr = (
    pd.read_csv(
        DATA_DIR / "EFFR.csv",
        parse_dates = ["observation_date"],
        date_format = "%d/%m/%Y",
        index_col = 0,
    ).squeeze()
    / 100
)

effr = effr.ffill() 

print(f"price data: {len(price_data):,} (date, ticker) rows")
print(f"dividend events: {len(dividend_data):,}")
print(f"SPY price observations: {len(daily_spy_returns):,}")
print(f"EFFR observations: {len(effr):,}")

# --- filter out bad tickers
price_data, bad_tickers = uc.remove_bad_tickers(price_data)

close_prices = price_data["close"]

all_tickers_during_backtest = (
    price_data.index.get_level_values("ticker").unique().to_list()
)

print(f"dropped {len(bad_tickers)} bad ticker(s)")

if not bad_tickers.empty:
    print(bad_tickers.head(10))
    
print(f"clean universe: {len(all_tickers_during_backtest)} tickers")

price data: 4,362,440 (date, ticker) rows
dividend events: 30,681
SPY price observations: 4,568
EFFR observations: 6,694
[INFO modules.universe_construction, remove_bad_tickers:287]  removing 84 ticker(s) with bad data (zero_volume=82, extreme_return=16)
dropped 84 bad ticker(s)
        zero_volume  extreme_return
ticker                             
ABI            True           False
AET            True           False
AMCR           True           False
AMD            True           False
ANDV           True           False
BHF            True           False
BKR            True           False
BMC            True            True
BOL            True            True
CA             True           False
clean universe: 871 tickers


## 2. Compute risk-free returns

In [4]:
r_d = effr / r.ACT_360  

calendar_days_since_previous_trading_date = (
    pd.Series(r_d.index, index=r_d.index)
    - pd.Series(r_d.index, index=r_d.index).shift(1)
).dt.days

rf_daily_returns = (
    (r_d.shift(1) * calendar_days_since_previous_trading_date)
    .rename("rf_return_since_previous_trading_date")
)
rf_daily_returns.index.name = "date"

print(f"rf returns: {rf_daily_returns.dropna().shape[0]:,} non-NaN observations")

rf_daily_returns.tail()

rf returns: 6,693 non-NaN observations


date
2026-02-20    0.000101
2026-02-23    0.000303
2026-02-24    0.000101
2026-02-25    0.000101
2026-02-26    0.000101
Name: rf_return_since_previous_trading_date, dtype: float64

## 3. Configure the backtest calendar

In [ ]:
# ---- Rebalance frequency and number of periods ----
start = pd.Timestamp("2023-01-05")  # automatically snapped backward to closest trading day, if needed
rebalance_freq_type = "ndays"     # "ndays" or "monthly"
rebalance_frequency_trading_days = 23
number_of_inter_rebalance_periods = 12  # covering ~1 calendar year (given the rebalance frequency of 21 trading days)

# ---- Estimation windows for portfolio-construction inputs  ----
rolling_periods_for_cov_matrix_estimation = 18
rolling_periods_for_beta_regression = 12
rolling_periods_for_avg_return_computation = 12
rolling_periods_for_factors_model_regression = 6

number_of_training_inter_rebalance_periods = max(
    rolling_periods_for_factors_model_regression,
    rolling_periods_for_cov_matrix_estimation,
    rolling_periods_for_beta_regression,
    rolling_periods_for_avg_return_computation
)

# ---- Trading-day rolling windows for daily features  ----
rolling_window_trading_days_for_volatility = 60
rolling_window_trading_days_for_momentum = 4 * 23   # ~4 trading months
buffer_trading_days_for_momentum = 23                # ~1 trading month skip
rolling_window_trading_days_for_adv = 60             # used only when capping universes


# ---- Build the calendar  ----
backtest_calendar = bcal.build_calendar(
                            start = start,
                            freq_type = rebalance_freq_type,
                            P = number_of_inter_rebalance_periods,
                            P_train = number_of_training_inter_rebalance_periods,
                            freq = rebalance_frequency_trading_days
                        )

scheduled_rebalance_dates = backtest_calendar.rebalance_dates
scheduled_training_rebalance_dates = backtest_calendar.training_rebalance_dates
backtest_dates = backtest_calendar.backtest_dates

print(
    f"backtest period: {backtest_dates[0]:%Y-%m-%d} to "
    f"{backtest_dates[-1]:%Y-%m-%d} ({len(backtest_dates)} trading days)"
)
print(
    f"rebalance dates: {len(scheduled_rebalance_dates)} "
    f"(training rebalance dates prepended: {len(scheduled_training_rebalance_dates) - len(scheduled_rebalance_dates)})"
)

# ---- Validate that we have enough data history ----
fetch_start, fetch_end = md.compute_price_data_window(
                                backtest_dates,
                                scheduled_training_rebalance_dates,
                                rolling_window_trading_days_for_momentum,
                                buffer_trading_days_for_momentum,
                                rolling_window_trading_days_for_volatility,
                                rolling_window_trading_days_for_adv
                            )

earliest_supported_date = pd.Timestamp("2008-01-01") # (see documentation, §6.2)
latest_supported_date = pd.Timestamp("2026-02-27")   # choose, at the latest, the last fetch date for price_data
if fetch_start < earliest_supported_date:
    raise ValueError(
        f"backtest needs data back to {fetch_start:%Y-%m-%d}, but earliest "
        f"supported date is {earliest_supported_date:%Y-%m-%d}. Choose a later start."
    )
if fetch_end > latest_supported_date:
    raise ValueError(
        f"backtest extends to {fetch_end:%Y-%m-%d}, but latest supported "
        f"date is {latest_supported_date:%Y-%m-%d}. Reduce P or pick an earlier start."
    )
print(f"data window required: {fetch_start:%Y-%m-%d} to {fetch_end:%Y-%m-%d}")

backtest period: 2023-01-05 to 2024-03-15 (300 trading days)
rebalance dates: 13 (training rebalance dates prepended: 18)
data window required: 2020-11-27 to 2024-03-18


## 4. Build per-rebalance-date universes

*(see documentation, §6)*


In [6]:
capping_num_tickers_per_universe = None  # set to None or to an integer (e.g. 200 to cap at top-200 by ADV)
if capping_num_tickers_per_universe is not None:
    price_data = uc.compute_adv(price_data, adv_window=rolling_window_trading_days_for_adv)


scheduled_reb_date_to_tickers_universe_dict, eligibility_criteria = (
                        uc.build_pit_universes(
                            scheduled_training_rebalance_dates,
                            final_period_end_date = backtest_dates[-1],
                            first_trading_day_of_month_to_sp500_members_dict =(
                                first_trading_day_of_month_to_sp500_members_dict
                            ),
                        price_data = price_data,
                        require_prices_for_next_period = True,
                        capping_num_tickers_per_universe = capping_num_tickers_per_universe,
                        )
)   


first_reb = scheduled_rebalance_dates[0]
last_reb = scheduled_rebalance_dates[-1]
print(
    f"universe size at first rebalance ({first_reb:%Y-%m-%d}): "
    f"{len(scheduled_reb_date_to_tickers_universe_dict[first_reb])}"
)
print(
    f"universe size at last rebalance ({last_reb:%Y-%m-%d}): "
    f"{len(scheduled_reb_date_to_tickers_universe_dict[last_reb])}"
)

print("\n head of dataframe of eligibility criteria: \n")

eligibility_criteria.head()

universe size at first rebalance (2023-01-05): 460
universe size at last rebalance (2024-02-12): 466

 head of dataframe of eligibility criteria: 



forward_close_prices_available  \
training_rebalance_date ticker                                   
2021-05-14              A                                 True   
                        AAL                               True   
                        AAP                               True   
                        AAPL                              True   
                        ABBV                              True   

                                forward_open_prices_available  
training_rebalance_date ticker                                 
2021-05-14              A                                True  
                        AAL                              True  
                        AAP                              True  
                        AAPL                             True  
                        ABBV                             True

# 5. Configure the optimization problem

In [7]:
# ---- Optimizer targets ----
targets = strat.MVOOptimizerTargets(
    objective = "min_variance",  # "max_return" or "min_variance"
    ptf_variance_cap = 0.04,    # (used when objective="max_return")
    ptf_return_lower_bound = 0.03,    # (used when objective="min_variance")
    ptf_beta_cap = 0.001,
    ptf_net_exposure_cap = 0.001,
    ptf_gross_exposure_cap = 2.0,
    max_shares_per_ticker = 10_000,
)



# --- the following is required for estimation method for expected returns ...:
# ... if using factors_model:
winsorize_factors_per_date = True
z_score_factors_per_date = True
use_ridge = False
ridge_penalty = 0.01 # (only used when use_ridge == True)
# ...if using ew_historical_average:
halflife_ew = 3




    
estimation = strat.MVOEstimationSettings(
    # "factors_model" | "regular_historical_average" | "ew_historical_average"
    next_period_expected_returns_estimation_method="factors_model",
    
    rolling_periods_for_factors_model_regression = rolling_periods_for_factors_model_regression,
    rolling_window_trading_days_for_momentum = rolling_window_trading_days_for_momentum,
    buffer_trading_days_for_momentum = buffer_trading_days_for_momentum,
    rolling_window_trading_days_for_volatility = rolling_window_trading_days_for_volatility,
    winsorize_factors_per_date = winsorize_factors_per_date,
    z_score_factors_per_date = z_score_factors_per_date,
    rolling_periods_for_avg_return_computation = rolling_periods_for_avg_return_computation,
    avg_type = "regular", # "regular" or "ew"
    halflife_ew = halflife_ew,
    rolling_periods_for_cov_matrix_estimation = rolling_periods_for_cov_matrix_estimation,
    rolling_periods_for_beta_regression = rolling_periods_for_beta_regression,
)

# ---- Precomputed inputs to the optimizer for each rebalance date  (for the fixed backtest rebalance schedule) 

 # estimation of expected returns 
if estimation.next_period_expected_returns_estimation_method == "factors_model":
    factors = fe.compute_factors_at_training_rebalance_dates(
            prices = close_prices,
            training_rebalance_dates = scheduled_training_rebalance_dates,
            rolling_window_trading_days_for_momentum = rolling_window_trading_days_for_momentum,
            buffer_trading_days_for_momentum = buffer_trading_days_for_momentum,
            rolling_window_trading_days_for_volatility = rolling_window_trading_days_for_volatility,
            reb_date_to_tickers_universe_dict = scheduled_reb_date_to_tickers_universe_dict,
            winsorize_factors_per_date = estimation.winsorize_factors_per_date,
            z_score_factors_per_date = estimation.z_score_factors_per_date,
    )

    last_backtest_date = backtest_dates[-1]
    
    realized_returns = rpm.compute_realized_period_returns(
            prices = close_prices,
            training_rebalance_dates = scheduled_training_rebalance_dates,
            final_period_end_date = last_backtest_date,
            reb_date_to_tickers_universe_dict = scheduled_reb_date_to_tickers_universe_dict                 
    )
    
    model_input = pd.concat([factors, realized_returns], axis=1, join="inner")
    
    factor_model_result = rpm.fit_predict_factors_model_at_rebalance_dates(
            rebalance_dates = scheduled_rebalance_dates,
            training_rebalance_dates = scheduled_training_rebalance_dates,
            model_input = model_input,
            rolling_periods = estimation.rolling_periods_for_factors_model_regression,    
            use_ridge = use_ridge,
            ridge_penalty = ridge_penalty
    )
    
    mu_predictions = factor_model_result.predictions
    
else:
    mu_predictions = rpm.rolling_mean_period_returns(
            prices = close_prices,
            training_rebalance_dates = scheduled_training_rebalance_dates,
            rebalance_dates = scheduled_rebalance_dates,
            rolling_number_of_periods = estimation.rolling_periods_for_avg_return_computation,
            avg_type = estimation.avg_type,
            halflife_ew = estimation.halflife_ew
    )

# estimation of covariance matrices
cov_result = cm.compute_rolling_cov_matrices(
        prices = close_prices,
        training_rebalance_dates = scheduled_training_rebalance_dates,
        rebalance_dates = scheduled_rebalance_dates,
        rolling_periods_for_estimation = estimation.rolling_periods_for_cov_matrix_estimation,
)
cov_matrices = cov_result.matrices

# estimation of betas
tickers_betas = bc.compute_capm_betas(
        rf_daily_returns = rf_daily_returns,
        market_daily_returns = daily_spy_returns,
        training_rebalance_dates = scheduled_training_rebalance_dates,
        rebalance_dates = scheduled_rebalance_dates,
        prices = close_prices,
        rolling_periods_for_beta_regression = estimation.rolling_periods_for_beta_regression
)

##  6. Solve the optimization problem at the selected backtest rebalance date

**Assemble optimzer's inputs:**

In [8]:
reb_date = scheduled_rebalance_dates[0] # selected anu date in `scheduled_rebalance_dates`

tickers = scheduled_reb_date_to_tickers_universe_dict[reb_date]

# data inputs for the optimizer for the chosen reb_date
mu = mu_predictions.xs(reb_date, level = 0)
sigma =  cov_matrices[reb_date]
betas = tickers_betas.xs(reb_date, level = 0)

**Run optimizer:**

In [9]:
common_kwargs = dict(
        ptf_beta_cap = targets.ptf_beta_cap,
        ptf_net_exposure_cap = targets.ptf_net_exposure_cap,
        ptf_gross_exposure_cap = targets.ptf_gross_exposure_cap,
    )



if targets.objective ==  "max_return":
   optimizer_result =  wo.optimize_weights(
                tickers, mu, sigma, betas,
                objective = "max_return",
                ptf_variance_cap = targets.ptf_variance_cap,
                **common_kwargs
            )

if targets.objective ==  "min_variance":
    
    if targets.ptf_return_lower_bound is None:
        raise ValueError("MVOOptimizerTargets.ptf_return_lower_bound is required when objective='min_variance'")

    halving_steps = 6 # select the desired number

    optimizer_result, _used_target = wo.optimize_min_variance_with_feasibility_recovery(
        tickers, mu, sigma, betas,
        initial_return_lower_bound = targets.ptf_return_lower_bound,
        min_return_lower_bound = targets.min_return_lower_bound_for_feasibility_recovery,
        halving_steps = halving_steps,
        **common_kwargs
    )

[WARNING modules.weights_optimizer, optimize_weights:207]  investable universe reduced from 460 to 459 tickers due to missing optimizer inputs (mu, sigma, or betas). Dropped: CEG
[INFO modules.weights_optimizer, optimize_min_variance_with_feasibility_recovery:435]  min-variance infeasible, halving return lower bound to 0.015000
[WARNING modules.weights_optimizer, optimize_weights:207]  investable universe reduced from 460 to 459 tickers due to missing optimizer inputs (mu, sigma, or betas). Dropped: CEG
[WARNING modules.weights_optimizer, optimize_min_variance_with_feasibility_recovery:425]  min-variance feasibility recovered after 1 halving step(s); used ptf_return_lower_bound = 0.015000 instead of requested 0.030000


## 7. Results

In [10]:
optimizer_metrics = optimizer_result.metrics
optimal_weights = optimizer_result.optimal_weights

**Optimization status**

In [11]:
optimizer_metrics["status"]

'optimal'

**Solvers tried**

In [12]:
optimizer_metrics["solvers_tried"]

['ECOS']

**Estimated optimal weights**

In [13]:
optimal_weights

A      -2.564809e-14
AAL    -1.136684e-15
AAP     2.485261e-15
AAPL   -1.030472e-14
ABBV   -1.197037e-02
            ...     
YUM    -3.517029e-02
ZBH    -2.385648e-14
ZBRA    9.510950e-15
ZION   -1.483899e-15
ZTS    -4.425530e-15
Name: optimal_weights, Length: 459, dtype: float64

**Number of tickers selected in the estimated  optimal portfolio**

In [14]:
len(optimizer_metrics["tickers"])

459

**Objective value at the estimated optimal weights:** this quantity represents either:

- the estimated expected return of the book’s equity, under simplifying assumptions 1–4 from the documentation (§9.1), when using the `max_returns` objective; or
- the estimated variance of the book’s equity when using the `min_variance` objective.

In both cases, the source of randomness is the future evolution of the prices of the tickers composing the portfolio.

In [15]:
optimizer_metrics["objective_value"]

0.001212970223537058

**Risk profile of the estimated optimal portfolio**

In [16]:
optimizer_metrics["risk"]

{'volatility': 0.03482772205495297,
 'variance': 0.0012129702235370576,
 'expected_return': 0.014999999999999888,
 'return_lower_bound': 0.015}

**Exposures of the estimated optimal portfolio (alongside their respective target)**

In [17]:
optimizer_metrics["exposures"]

{'net': 0.0010000000000016662,
 'net_cap': 0.001,
 'gross': 1.9999999999950118,
 'gross_cap': 2.0,
 'beta': 0.0009999999999132133,
 'beta_cap': 0.001}

**Residuals between actual exposures and their targets**

In [18]:
optimizer_metrics["residuals"]

{'net_minus_cap': 1.6662018986757232e-15,
 'gross_minus_cap': -4.988232049640828e-12,
 'beta_minus_epsilon': -8.678669762007818e-14,
 'return_minus_lower_bound': -1.1102230246251565e-16}

All residuals are effectively zero (given the floating-point arithmetic), confirming the constraints are closely satisfied by the estimated optimal portfolio.